In [1]:
# 1. Import thư viện
import os
import pickle

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error, mean_absolute_error

from scipy.sparse import vstack

from surprise import Dataset, Reader, SVD


In [2]:
# !pip install scikit-surprise


In [3]:
# 2. Đọc dữ liệu

ratings_df = pd.read_csv("data/ratings.csv")
movies_df = pd.read_csv("data/movies.csv")
tags_df = pd.read_csv("data/tags.csv")
links_df = pd.read_csv("data/links.csv")

print("Ratings:", ratings_df.shape)
print("Movies:", movies_df.shape)
print("Tags:", tags_df.shape)
print("Links:", links_df.shape)


Ratings: (100836, 4)
Movies: (9742, 3)
Tags: (3683, 4)
Links: (9742, 3)


In [4]:
ratings_df.info()
ratings_df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [5]:
movies_df.info()
movies_df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  9742 non-null   int64 
 1   title    9742 non-null   object
 2   genres   9742 non-null   object
dtypes: int64(1), object(2)
memory usage: 228.5+ KB


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [6]:
tags_df.info()
tags_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3683 entries, 0 to 3682
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   userId     3683 non-null   int64 
 1   movieId    3683 non-null   int64 
 2   tag        3683 non-null   object
 3   timestamp  3683 non-null   int64 
dtypes: int64(3), object(1)
memory usage: 115.2+ KB


,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


In [7]:
# 3. Kiểm tra dữ liệu
#3.1. Kiểm tra dữ liệu trùng lặp
print("Ratings duplicate:", ratings_df.duplicated().sum())
print("Movies duplicate:", movies_df.duplicated().sum())
print("Tags duplicate:", tags_df.duplicated().sum())


Ratings duplicate: 0
Movies duplicate: 0
Tags duplicate: 0


In [8]:
#3.2. Chuẩn bị dữ liệu rating cho SVD
ratings_svd = ratings_df[
    ["userId", "movieId", "rating"]
].copy()

ratings_svd.info()
ratings_svd.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 3 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   userId   100836 non-null  int64  
 1   movieId  100836 non-null  int64  
 2   rating   100836 non-null  float64
dtypes: float64(1), int64(2)
memory usage: 2.3 MB


,userId,movieId,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0


In [9]:
ratings_svd["rating"].describe()

count    100836.000000
mean          3.501557
std           1.042529
min           0.500000
25%           3.000000
50%           3.500000
75%           4.000000
max           5.000000
Name: rating, dtype: float64

In [10]:
n_users = ratings_svd["userId"].nunique()
n_movies = ratings_svd["movieId"].nunique()
n_ratings = len(ratings_svd)

print("Số user:", n_users)
print("Số phim có rating:", n_movies)
print("Số lượt rating:", n_ratings)

sparsity = (
    1 - n_ratings / (n_users * n_movies)
) * 100

print(f"Sparsity: {sparsity:.2f}%")


Số user: 610
Số phim có rating: 9724
Số lượt rating: 100836
Sparsity: 98.30%


In [11]:
# 4. Tiền xử lý dữ liệu
# 4.1. Tách năm phát hành từ title
movies_df["year"] = movies_df["title"].str.extract(r"\((\d{4})\)")

movies_df["year"] = pd.to_numeric(
    movies_df["year"],
    errors="coerce"
)

movies_df.head()

,movieId,title,genres,year
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995.0
2,3,Grumpier Old Men (1995),Comedy|Romance,1995.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995.0
4,5,Father of the Bride Part II (1995),Comedy,1995.0


In [12]:
# 4.2. Xóa năm khỏi title
movies_df["title"] = (
    movies_df["title"]
    .str.replace(r"\(\d{4}\)", "", regex=True)
    .str.strip()
)

movies_df.head()

,movieId,title,genres,year
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995.0
1,2,Jumanji,Adventure|Children|Fantasy,1995.0
2,3,Grumpier Old Men,Comedy|Romance,1995.0
3,4,Waiting to Exhale,Comedy|Drama|Romance,1995.0
4,5,Father of the Bride Part II,Comedy,1995.0


In [13]:
# 4.3. Chuẩn hóa genres
count_no_genres = len(
    movies_df[movies_df["genres"] == "(no genres listed)"]
)

print("Số phim không có genres:", count_no_genres)

movies_df["genres"] = movies_df["genres"].replace(
    "(no genres listed)",
    ""
)

movies_df["genres"] = movies_df["genres"].str.replace(
    "|",
    " ",
    regex=False
)

movies_df.head()


Số phim không có genres: 34


,movieId,title,genres,year
0,1,Toy Story,Adventure Animation Children Comedy Fantasy,1995.0
1,2,Jumanji,Adventure Children Fantasy,1995.0
2,3,Grumpier Old Men,Comedy Romance,1995.0
3,4,Waiting to Exhale,Comedy Drama Romance,1995.0
4,5,Father of the Bride Part II,Comedy,1995.0


In [14]:
# 4.4. Chuẩn hóa tag
tags_df["tag"] = (
    tags_df["tag"]
    .astype(str)
    .str.lower()
    .str.strip()
)

tags_df.head()


,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,boxing story,1445715207
4,2,89774,mma,1445715200


In [15]:
# 4.5. Gom tag theo từng phim
movie_tags = (
    tags_df
    .groupby("movieId")["tag"]
    .apply(" ".join)
    .reset_index()
)

movie_tags.head()


,movieId,tag
0,1,pixar pixar fun
1,2,fantasy magic board game robin williams game
2,3,moldy old
3,5,pregnancy remake
4,7,remake


In [16]:
# 4.7. Ghép tag vào dữ liệu phim
movies_nlp = movies_df.merge(
    movie_tags,
    on="movieId",
    how="left"
)

movies_nlp["tag"] = movies_nlp["tag"].fillna("")

movies_nlp.head()


,movieId,title,genres,year,tag
0,1,Toy Story,Adventure Animation Children Comedy Fantasy,1995.0,pixar pixar fun
1,2,Jumanji,Adventure Children Fantasy,1995.0,fantasy magic board game robin williams game
2,3,Grumpier Old Men,Comedy Romance,1995.0,moldy old
3,4,Waiting to Exhale,Comedy Drama Romance,1995.0,
4,5,Father of the Bride Part II,Comedy,1995.0,pregnancy remake


In [17]:
# 4.8. Tạo cột content
movies_nlp["content"] = (
    movies_nlp["title"]
    + " "
    + movies_nlp["genres"]
    + " "
    + movies_nlp["tag"]
)

movies_nlp.head()


,movieId,title,genres,year,tag,content
0,1,Toy Story,Adventure Animation Children Comedy Fantasy,1995.0,pixar pixar fun,Toy Story Adventure Animation Children Comedy ...
1,2,Jumanji,Adventure Children Fantasy,1995.0,fantasy magic board game robin williams game,Jumanji Adventure Children Fantasy fantasy mag...
2,3,Grumpier Old Men,Comedy Romance,1995.0,moldy old,Grumpier Old Men Comedy Romance moldy old
3,4,Waiting to Exhale,Comedy Drama Romance,1995.0,,Waiting to Exhale Comedy Drama Romance
4,5,Father of the Bride Part II,Comedy,1995.0,pregnancy remake,Father of the Bride Part II Comedy pregnancy r...


In [18]:
# 4.9. Làm sạch content
movies_nlp["content"] = (
    movies_nlp["content"]
    .str.lower()
)

special_content = movies_nlp[
    movies_nlp["content"].str.contains(
        r"[^a-zA-Z0-9\s]",
        regex=True,
        na=False
    )
]

print("Số dòng content có ký tự đặc biệt:", len(special_content))

movies_nlp["content"] = (
    movies_nlp["content"]
    .str.replace(
        r"[^a-zA-Z0-9\s]",
        " ",
        regex=True
    )
)

movies_nlp["content"] = (
    movies_nlp["content"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

movies_nlp.head()


Số dòng content có ký tự đặc biệt: 4858


,movieId,title,genres,year,tag,content
0,1,Toy Story,Adventure Animation Children Comedy Fantasy,1995.0,pixar pixar fun,toy story adventure animation children comedy ...
1,2,Jumanji,Adventure Children Fantasy,1995.0,fantasy magic board game robin williams game,jumanji adventure children fantasy fantasy mag...
2,3,Grumpier Old Men,Comedy Romance,1995.0,moldy old,grumpier old men comedy romance moldy old
3,4,Waiting to Exhale,Comedy Drama Romance,1995.0,,waiting to exhale comedy drama romance
4,5,Father of the Bride Part II,Comedy,1995.0,pregnancy remake,father of the bride part ii comedy pregnancy r...


In [19]:
# 5. Trích chọn đặc trưng TF-IDF
tfidf = TfidfVectorizer(
    stop_words="english"
)

tfidf_matrix = tfidf.fit_transform(
    movies_nlp["content"]
)

print("Kích thước ma trận TF-IDF:", tfidf_matrix.shape)

feature_names = tfidf.get_feature_names_out()

print("200 từ đầu tiên trong từ vựng:")
print(feature_names[:200])


Kích thước ma trận TF-IDF: (9742, 9801)
200 từ đầu tiên trong từ vựng:
['00' '000' '007' '01' '04' '06' '09' '10' '100' '1000' '101' '102' '10th'
 '11' '1138' '11th' '12' '120' '127' '13' '13th' '14' '1408' '1492' '15'
 '16' '1600' '17' '174' '1776' '18' '187' '19' '1900' '1900s' '1920s'
 '1933' '1935' '1941' '1950s' '1960s' '1969' '1970s' '1972' '1975' '1980s'
 '1984' '1985' '1990' '1990s' '1992' '1st' '20' '200' '2000' '2001' '2006'
 '2007' '2010' '2012' '2018' '2046' '2048' '2049' '21' '211' '22' '23'
 '24' '25' '250' '25th' '27' '28' '281' '2d' '2nd' '30' '300' '3000' '31'
 '33' '34th' '35' '37th' '39' '3d' '3dd' '40' '400' '42' '42nd' '43' '44'
 '45' '451' '46' '47' '48' '49' '4th' '50' '500' '51' '52' '54' '57' '571'
 '5th' '60' '61' '66' '6th' '70' '70mm' '71' '73' '77' '777' '7th' '80'
 '800' '81' '84' '8mm' '8th' '90' '900' '911' '93' '96' '964' '99' '9to5'
 'aan' 'aardman' 'aaron' 'abandoned' 'abbey' 'abbott' 'abbotts' 'abcs'
 'abduction' 'abiding' 'abominable' 'abortion' 'ab

In [20]:
# 6. Chia train/dev/test
ratings_train, ratings_temp = train_test_split(
    ratings_svd,
    test_size=0.30,
    random_state=42
)

ratings_dev, ratings_test = train_test_split(
    ratings_temp,
    test_size=0.50,
    random_state=42
)

print("--- KÍCH THƯỚC DỮ LIỆU SAU KHI CHIA ---")
print("Train:", ratings_train.shape)
print("Dev:", ratings_dev.shape)
print("Test:", ratings_test.shape)

print("Tỷ lệ Train:", len(ratings_train) / len(ratings_svd))
print("Tỷ lệ Dev:", len(ratings_dev) / len(ratings_svd))
print("Tỷ lệ Test:", len(ratings_test) / len(ratings_svd))


--- KÍCH THƯỚC DỮ LIỆU SAU KHI CHIA ---
Train: (70585, 3)
Dev: (15125, 3)
Test: (15126, 3)
Tỷ lệ Train: 0.6999980165813796
Tỷ lệ Dev: 0.14999603316275933
Tỷ lệ Test: 0.150005950255861


In [21]:
# 7. Huấn luyện SVD
reader = Reader(
    rating_scale=(0.5, 5.0)
)

data_surprise = Dataset.load_from_df(
    ratings_train[["userId", "movieId", "rating"]],
    reader
)

trainset = data_surprise.build_full_trainset()

svd_model = SVD(
    n_factors=50,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

svd_model.fit(trainset)

print("Đã huấn luyện xong mô hình SVD.")


Đã huấn luyện xong mô hình SVD.


In [22]:
print("--- KẾT QUẢ TRÍCH CHỌN ĐẶC TRƯNG SVD ---")

user_features = svd_model.pu
movie_features = svd_model.qi

print(f"Kích thước ma trận đặc trưng ẩn Người dùng (P): {user_features.shape}")
print(f"Kích thước ma trận đặc trưng ẩn Bộ phim (Q): {movie_features.shape}")

print("Vector đặc trưng ẩn của User đầu tiên - 10 chiều đầu:")
print(user_features[0][:10])

print("Vector đặc trưng ẩn của Movie đầu tiên - 10 chiều đầu:")
print(movie_features[0][:10])


--- KẾT QUẢ TRÍCH CHỌN ĐẶC TRƯNG SVD ---
Kích thước ma trận đặc trưng ẩn Người dùng (P): (610, 50)
Kích thước ma trận đặc trưng ẩn Bộ phim (Q): (8566, 50)
Vector đặc trưng ẩn của User đầu tiên - 10 chiều đầu:
[-0.15365916  0.01390026  0.03460255  0.37580732 -0.09834527 -0.05371136
  0.14063207  0.23963056 -0.07833458 -0.02618404]
Vector đặc trưng ẩn của Movie đầu tiên - 10 chiều đầu:
[ 0.04215958  0.13717139 -0.0177633  -0.10364534  0.04064962 -0.06499308
 -0.04216886  0.06882788  0.12584783  0.05473791]


In [23]:
# 8. Đánh giá SVD bằng RMSE và MAE
required_variables = [
    "ratings_svd",
    "ratings_train",
    "ratings_dev",
    "ratings_test",
    "movies_nlp",
    "tfidf_matrix",
    "svd_model"
]

for var_name in required_variables:
    if var_name not in globals():
        raise NameError(
            f"Bạn cần chạy các cell phía trên trước. Biến còn thiếu: {var_name}"
        )

print("Các biến cần thiết đã sẵn sàng.")
print("ratings_train:", ratings_train.shape)
print("ratings_dev:", ratings_dev.shape)
print("ratings_test:", ratings_test.shape)
print("movies_nlp:", movies_nlp.shape)
print("tfidf_matrix:", tfidf_matrix.shape)


Các biến cần thiết đã sẵn sàng.
ratings_train: (70585, 3)
ratings_dev: (15125, 3)
ratings_test: (15126, 3)
movies_nlp: (9742, 6)
tfidf_matrix: (9742, 9801)


In [24]:
def evaluate_svd_model(model, ratings_df):
    y_true = []
    y_pred = []

    for row in ratings_df.itertuples():
        user_id = row.userId
        movie_id = row.movieId
        true_rating = row.rating

        predicted_rating = model.predict(
            user_id,
            movie_id
        ).est

        y_true.append(true_rating)
        y_pred.append(predicted_rating)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    return rmse, mae


In [25]:
dev_rmse, dev_mae = evaluate_svd_model(
    svd_model,
    ratings_dev
)

test_rmse, test_mae = evaluate_svd_model(
    svd_model,
    ratings_test
)

svd_error_results = pd.DataFrame([
    {
        "Dataset": "Dev",
        "RMSE": dev_rmse,
        "MAE": dev_mae
    },
    {
        "Dataset": "Test",
        "RMSE": test_rmse,
        "MAE": test_mae
    }
])

svd_error_results


,Dataset,RMSE,MAE
0,Dev,0.877207,0.670134
1,Test,0.881863,0.680190


In [26]:
# 9. Xây dựng Hybrid Recommendation
# 9.1. Cấu hình chung cho Hybrid
# RELEVANCE_THRESHOLD = 3.5: rating từ 3.5 trở lên được xem là phim user thích.
# K = 10: đánh giá Top-10 phim.
# ALPHA_LIST = [0.5, 0.7, 0.9]: các giá trị alpha để thử trên tập dev.

RELEVANCE_THRESHOLD = 3.5
K = 10
ALPHA_LIST = [0.5, 0.7, 0.9]


In [27]:
# 9.2. Tạo các biến tra cứu
# user_train_items: lưu các phim mà mỗi user đã rating trong tập train.
# movie_id_to_index: ánh xạ từ `movieId` sang index trong `tfidf_matrix`

user_train_items = (
    ratings_train
    .groupby("userId")["movieId"]
    .apply(set)
    .to_dict()
)

movie_id_to_index = pd.Series(
    movies_nlp.index,
    index=movies_nlp["movieId"]
).to_dict()

print("Số user trong Train:", len(user_train_items))
print("Số phim trong movie_id_to_index:", len(movie_id_to_index))


Số user trong Train: 610
Số phim trong movie_id_to_index: 9742


In [28]:
# 9.3. Chuẩn hóa điểm SVD (0-1)
def normalize_svd_score(score, min_rating=0.5, max_rating=5.0):
    normalized_score = (score - min_rating) / (max_rating - min_rating)
    return max(0, min(1, normalized_score))

In [29]:
# 9.4. Xây dựng hồ sơ sở thích người dùng
def build_user_profile(
    user_id,
    ratings_df,
    threshold=3.5
):
    liked_ratings = ratings_df[
        (ratings_df["userId"] == user_id) &
        (ratings_df["rating"] >= threshold)
    ]

    if liked_ratings.empty:
        return None

    movie_vectors = []
    weights = []

    for row in liked_ratings.itertuples():
        movie_id = row.movieId

        if movie_id in movie_id_to_index:
            movie_index = movie_id_to_index[movie_id]
            movie_vectors.append(tfidf_matrix[movie_index])
            weights.append(row.rating)

    if len(movie_vectors) == 0:
        return None

    movie_matrix = vstack(movie_vectors)
    weights = np.array(weights)

    user_profile = (
        movie_matrix.multiply(weights[:, None]).sum(axis=0)
        / weights.sum()
    )

    return np.asarray(user_profile)


In [30]:
# 9.5. Hàm gợi ý phim cho một user
def recommend_movies_for_user(
    user_id,
    alpha=0.7,
    top_n=10,
    threshold=3.5,
    method="hybrid"
):
    rated_items = user_train_items.get(user_id, set())

    candidate_movies = movies_nlp[
        ~movies_nlp["movieId"].isin(rated_items)
    ].copy()

    user_profile = build_user_profile(
        user_id=user_id,
        ratings_df=ratings_train,
        threshold=threshold
    )

    if user_profile is not None:
        content_scores = cosine_similarity(
            user_profile,
            tfidf_matrix
        ).flatten()
    else:
        content_scores = None

    recommendations = []

    for row in candidate_movies.itertuples():
        movie_id = row.movieId

        svd_pred_rating = svd_model.predict(
            user_id,
            movie_id
        ).est

        svd_score_norm = normalize_svd_score(
            svd_pred_rating
        )

        if content_scores is not None and movie_id in movie_id_to_index:
            content_score = content_scores[
                movie_id_to_index[movie_id]
            ]
        else:
            content_score = 0

        hybrid_score = (
            alpha * svd_score_norm
            + (1 - alpha) * content_score
        )

        if method == "svd":
            final_score = svd_score_norm
        elif method == "hybrid":
            final_score = hybrid_score
        else:
            raise ValueError("method phải là 'svd' hoặc 'hybrid'.")

        recommendations.append({
            "movieId": movie_id,
            "title": row.title,
            "genres": row.genres,
            "svd_pred_rating": svd_pred_rating,
            "svd_score_norm": svd_score_norm,
            "content_score": content_score,
            "hybrid_score": hybrid_score,
            "final_score": final_score
        })

    recommendations_df = pd.DataFrame(recommendations)

    if recommendations_df.empty:
        return pd.DataFrame({
            "message": [f"Không còn phim phù hợp để gợi ý cho user {user_id}."]
        })

    return (
        recommendations_df
        .sort_values("final_score", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )


In [31]:
# 10. Đánh giá Top-K
# 10.1. Lấy phim phù hợp trong dev/test
def get_relevant_items(ratings_df, threshold=3.5):
    relevant_df = ratings_df[
        ratings_df["rating"] >= threshold
    ]

    relevant_items = (
        relevant_df
        .groupby("userId")["movieId"]
        .apply(set)
        .to_dict()
    )

    return relevant_items


In [32]:
# 10.2. Tính Precision@K, Recall@K, NDCG@K
def precision_recall_ndcg_at_k(
    recommended_items,
    relevant_items,
    k=10
):
    recommended_items = recommended_items[:k]

    if len(relevant_items) == 0:
        return None, None, None

    hits = [
        1 if movie_id in relevant_items else 0
        for movie_id in recommended_items
    ]

    precision = sum(hits) / k
    recall = sum(hits) / len(relevant_items)

    dcg = 0
    for i, hit in enumerate(hits):
        dcg += hit / np.log2(i + 2)

    ideal_hits = [1] * min(len(relevant_items), k)

    idcg = 0
    for i, hit in enumerate(ideal_hits):
        idcg += hit / np.log2(i + 2)

    ndcg = dcg / idcg if idcg > 0 else 0

    return precision, recall, ndcg


In [33]:
# 10.3. Hàm đánh giá Top-K
def evaluate_topk_model(
    ratings_eval,
    method="hybrid",
    alpha=0.7,
    k=10,
    threshold=3.5
):
    relevant_items_dict = get_relevant_items(
        ratings_eval,
        threshold=threshold
    )

    precision_list = []
    recall_list = []
    ndcg_list = []

    for user_id, relevant_items in relevant_items_dict.items():

        recommendations = recommend_movies_for_user(
            user_id=user_id,
            alpha=alpha,
            top_n=k,
            threshold=threshold,
            method=method
        )

        if "message" in recommendations.columns:
            continue

        recommended_items = recommendations["movieId"].tolist()

        precision, recall, ndcg = precision_recall_ndcg_at_k(
            recommended_items=recommended_items,
            relevant_items=relevant_items,
            k=k
        )

        if precision is not None:
            precision_list.append(precision)
            recall_list.append(recall)
            ndcg_list.append(ndcg)

    return {
        "Model": method,
        "Alpha": alpha if method == "hybrid" else "-",
        f"Precision@{k}": np.mean(precision_list),
        f"Recall@{k}": np.mean(recall_list),
        f"NDCG@{k}": np.mean(ndcg_list),
        "Number of Users": len(precision_list)
    }


In [34]:
# 10.4. Đánh giá trên Dev để chọn alpha
dev_results = []

dev_results.append(
    evaluate_topk_model(
        ratings_eval=ratings_dev,
        method="svd",
        k=K,
        threshold=RELEVANCE_THRESHOLD
    )
)

for alpha in ALPHA_LIST:
    dev_results.append(
        evaluate_topk_model(
            ratings_eval=ratings_dev,
            method="hybrid",
            alpha=alpha,
            k=K,
            threshold=RELEVANCE_THRESHOLD
        )
    )

dev_results_df = pd.DataFrame(dev_results)

dev_results_df


,Model,Alpha,Precision@10,Recall@10,NDCG@10,Number of Users
0,svd,-,0.036949,0.033438,0.049547,590
1,hybrid,0.5,0.016610,0.022938,0.024275,590
2,hybrid,0.7,0.040339,0.045888,0.052762,590
3,hybrid,0.9,0.042373,0.041131,0.057342,590


In [35]:
hybrid_dev_results = dev_results_df[
    dev_results_df["Model"] == "hybrid"
].copy()

best_row = hybrid_dev_results.sort_values(
    by=f"NDCG@{K}",
    ascending=False
).iloc[0]

best_alpha = float(best_row["Alpha"])

print("Alpha tốt nhất theo NDCG@10 trên tập Dev:", best_alpha)


Alpha tốt nhất theo NDCG@10 trên tập Dev: 0.9


In [36]:
# 10.5. Đánh giá cuối cùng trên Test
test_results = []

test_results.append(
    evaluate_topk_model(
        ratings_eval=ratings_test,
        method="svd",
        k=K,
        threshold=RELEVANCE_THRESHOLD
    )
)

test_results.append(
    evaluate_topk_model(
        ratings_eval=ratings_test,
        method="hybrid",
        alpha=best_alpha,
        k=K,
        threshold=RELEVANCE_THRESHOLD
    )
)

test_results_df = pd.DataFrame(test_results)

test_results_df


,Model,Alpha,Precision@10,Recall@10,NDCG@10,Number of Users
0,svd,-,0.036949,0.028962,0.046287,590
1,hybrid,0.9,0.044068,0.039315,0.054154,590


In [37]:
# 11. Demo gợi ý
# 11.1. Hàm thống kê thể loại user yêu thích
def get_user_favorite_genres(
    user_id,
    ratings_df,
    movies_df,
    min_rating=3.5
):
    user_ratings = ratings_df[
        (ratings_df["userId"] == user_id) &
        (ratings_df["rating"] >= min_rating)
    ].copy()

    if user_ratings.empty:
        return pd.DataFrame(
            columns=["genre", "count", "avg_rating"]
        )

    user_movies = user_ratings.merge(
        movies_df[["movieId", "genres"]],
        on="movieId",
        how="left"
    )

    genre_stats = {}

    for row in user_movies.itertuples():
        genres = str(row.genres).split()

        for genre in genres:
            if genre not in genre_stats:
                genre_stats[genre] = {
                    "ratings": [],
                    "count": 0
                }

            genre_stats[genre]["ratings"].append(row.rating)
            genre_stats[genre]["count"] += 1

    result = []

    for genre, stats in genre_stats.items():
        result.append({
            "genre": genre,
            "count": stats["count"],
            "avg_rating": np.mean(stats["ratings"])
        })

    return (
        pd.DataFrame(result)
        .sort_values(
            by=["avg_rating", "count"],
            ascending=False
        )
        .reset_index(drop=True)
    )


In [38]:
# 11.2. Demo với user cụ thể
demo_user_id = 1

favorite_genres = get_user_favorite_genres(
    user_id=demo_user_id,
    ratings_df=ratings_train,
    movies_df=movies_nlp,
    min_rating=RELEVANCE_THRESHOLD
)

favorite_genres.head(10)


,genre,count,avg_rating
0,Mystery,10,4.800000
1,Musical,14,4.785714
2,War,16,4.625000
3,Animation,13,4.615385
4,Children,23,4.608696
5,Drama,49,4.591837
6,Crime,30,4.566667
7,Adventure,53,4.547170
8,Action,55,4.527273
9,Comedy,57,4.526316


In [39]:
demo_recommendations = recommend_movies_for_user(
    user_id=demo_user_id,
    alpha=best_alpha,
    top_n=10,
    threshold=RELEVANCE_THRESHOLD,
    method="hybrid"
)

demo_recommendations[
    [
        "movieId",
        "title",
        "genres",
        "svd_pred_rating",
        "svd_score_norm",
        "content_score",
        "hybrid_score"
    ]
]


,movieId,title,genres,svd_pred_rating,svd_score_norm,content_score,hybrid_score
0,1223,"Grand Day Out with Wallace and Gromit, A",Adventure Animation Children Comedy Sci-Fi,5.000000,1.000000,0.238932,0.923893
1,1262,"Great Escape, The",Action Adventure Drama War,5.000000,1.000000,0.225045,0.922504
2,78499,Toy Story 3,Adventure Animation Children Comedy Fantasy IMAX,4.961318,0.991404,0.298108,0.922075
3,48774,Children of Men,Action Adventure Drama Sci-Fi Thriller,4.829447,0.962099,0.554966,0.921386
4,4011,Snatch,Comedy Crime Thriller,4.975131,0.994474,0.177160,0.912742
5,750,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy War,4.999205,0.999823,0.118509,0.911692
6,858,"Godfather, The",Crime Drama,5.000000,1.000000,0.109523,0.910952
7,1201,"Good, the Bad and the Ugly, The (Buono, il bru...",Action Adventure Western,5.000000,1.000000,0.089164,0.908916
8,6016,City of God (Cidade de Deus),Action Adventure Crime Drama Thriller,4.944798,0.987733,0.180038,0.906963
9,904,Rear Window,Mystery Thriller,5.000000,1.000000,0.057563,0.905756


In [40]:
# 11.3. So sánh gợi ý với nhiều alpha
for alpha in ALPHA_LIST:
    print("=" * 80)
    print(f"Top 5 phim gợi ý cho user {demo_user_id} với alpha = {alpha}")

    result = recommend_movies_for_user(
        user_id=demo_user_id,
        alpha=alpha,
        top_n=5,
        threshold=RELEVANCE_THRESHOLD,
        method="hybrid"
    )

    display(
        result[
            [
                "title",
                "genres",
                "svd_pred_rating",
                "svd_score_norm",
                "content_score",
                "hybrid_score"
            ]
        ]
    )


Top 5 phim gợi ý cho user 1 với alpha = 0.5


,title,genres,svd_pred_rating,svd_score_norm,content_score,hybrid_score
0,Children of Men,Action Adventure Drama Sci-Fi Thriller,4.829447,0.962099,0.554966,0.758533
1,Eight Below,Action Adventure Drama Romance,4.108405,0.801868,0.555026,0.678447
2,Now You See Me 2,Action Comedy Thriller,4.334482,0.852107,0.486440,0.669274
3,9,Adventure Animation Sci-Fi,4.358317,0.857404,0.446623,0.652013
4,D.A.R.Y.L.,Adventure Children Sci-Fi,4.153957,0.811990,0.486672,0.649331


Top 5 phim gợi ý cho user 1 với alpha = 0.7


,title,genres,svd_pred_rating,svd_score_norm,content_score,hybrid_score
0,Children of Men,Action Adventure Drama Sci-Fi Thriller,4.829447,0.962099,0.554966,0.839959
1,Toy Story 3,Adventure Animation Children Comedy Fantasy IMAX,4.961318,0.991404,0.298108,0.783415
2,"Grand Day Out with Wallace and Gromit, A",Adventure Animation Children Comedy Sci-Fi,5.000000,1.000000,0.238932,0.771679
3,"Great Escape, The",Action Adventure Drama War,5.000000,1.000000,0.225045,0.767513
4,"Last Detail, The",Comedy Drama,4.663948,0.925322,0.371308,0.759118


Top 5 phim gợi ý cho user 1 với alpha = 0.9


,title,genres,svd_pred_rating,svd_score_norm,content_score,hybrid_score
0,"Grand Day Out with Wallace and Gromit, A",Adventure Animation Children Comedy Sci-Fi,5.000000,1.000000,0.238932,0.923893
1,"Great Escape, The",Action Adventure Drama War,5.000000,1.000000,0.225045,0.922504
2,Toy Story 3,Adventure Animation Children Comedy Fantasy IMAX,4.961318,0.991404,0.298108,0.922075
3,Children of Men,Action Adventure Drama Sci-Fi Thriller,4.829447,0.962099,0.554966,0.921386
4,Snatch,Comedy Crime Thriller,4.975131,0.994474,0.177160,0.912742


In [41]:
# 12. Lưu kết quả và model
svd_error_results.to_csv(
    "data/svd_error_results.csv",
    index=False
)

dev_results_df.to_csv(
    "data/evaluation_dev_results.csv",
    index=False
)

test_results_df.to_csv(
    "data/evaluation_test_results.csv",
    index=False
)

movies_nlp.to_csv(
    "data/movies_nlp_processed.csv",
    index=False
)

print("Đã lưu các file kết quả vào thư mục data.")


Đã lưu các file kết quả vào thư mục data.


In [42]:
# 12.1. Lưu model cho Streamlit
os.makedirs("movie_rcm_demo/model", exist_ok=True)

with open("movie_rcm_demo/model/svd_model.pkl", "wb") as f:
    pickle.dump(svd_model, f)

with open("movie_rcm_demo/model/tfidf_matrix.pkl", "wb") as f:
    pickle.dump(tfidf_matrix, f)

with open("movie_rcm_demo/model/movies_nlp.pkl", "wb") as f:
    pickle.dump(movies_nlp, f)

with open("movie_rcm_demo/model/ratings_train.pkl", "wb") as f:
    pickle.dump(ratings_train, f)

with open("movie_rcm_demo/model/movie_id_to_index.pkl", "wb") as f:
    pickle.dump(movie_id_to_index, f)

with open("movie_rcm_demo/model/best_alpha.pkl", "wb") as f:
    pickle.dump(best_alpha, f)

print("Đã lưu model vào movie_rcm_demo/model")


Đã lưu model vào movie_rcm_demo/model
